<a href="https://colab.research.google.com/github/orvencasido/ai-chatbot/blob/main/chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch pandas

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments
import torch

# Load GPT-2 pre-trained model and tokenizer
model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Tokenizer setup
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Prepare a list of question-answer pairs from the data
qa_pairs = [
    {"question": "What is your name?", "answer": "My name is Orven Casido."},
]

In [ ]:
def encode_data(qa_pairs, max_length=512):
    # Encode the QA pairs into GPT-2 format with padding
    inputs = []
    for qa in qa_pairs:
        question = qa['question']
        answer = qa['answer']
        text = f"Q: {question} A: {answer}"

        # Tokenize with padding and truncation to ensure all sequences are the same length
        encodings = tokenizer.encode(text, truncation=True, padding='max_length', max_length=max_length)

        # Return input_ids and labels (for causal language modeling, labels are the same as input_ids)
        inputs.append({
            'input_ids': encodings,
            'labels': encodings  # For language modeling, labels = input_ids
        })

    return inputs

In [ ]:
# Encode data
encoded_data = encode_data(qa_pairs)

# Convert to tensors (separate input_ids and labels)
input_ids = torch.tensor([item['input_ids'] for item in encoded_data])
labels = torch.tensor([item['labels'] for item in encoded_data])

# Create a TensorDataset that includes both input_ids and labels
train_dataset = torch.utils.data.TensorDataset(input_ids, labels)

In [ ]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, encoded_data):
        self.input_ids = torch.tensor([item['input_ids'] for item in encoded_data])
        self.labels = torch.tensor([item['labels'] for item in encoded_data])

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'labels': self.labels[idx]
        }

# Encode the data
encoded_data = encode_data(qa_pairs)

# Create the custom dataset
train_dataset = CustomDataset(encoded_data)

In [ ]:
# Set up the training arguments
training_args = TrainingArguments(
    output_dir='/content/results',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    save_steps=10_000,
    save_total_limit=2,
)

# Set up the Trainer with the custom dataset
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

# Start training
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5, training_loss=3.2322017669677736, metrics={'train_runtime': 67.8276, 'train_samples_per_second': 0.074, 'train_steps_per_second': 0.074, 'total_flos': 1306460160000.0, 'train_loss': 3.2322017669677736, 'epoch': 5.0})

In [ ]:
# Save the fine-tuned model
model.save_pretrained('/content/fine_tuned_model')
tokenizer.save_pretrained('/content/fine_tuned_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/fine_tuned_model/tokenizer_config.json',
 '/content/fine_tuned_model/tokenizer.json')

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load the fine-tuned model and tokenizer
model = GPT2LMHeadModel.from_pretrained('/content/fine_tuned_model')
tokenizer = GPT2Tokenizer.from_pretrained('/content/fine_tuned_model')

# Define a function to ask the model questions
def ask_ai(question):
    # Tokenize the input question
    inputs = tokenizer(question, return_tensors='pt', padding=True, truncation=True, max_length=512)

    # Generate the output from the model
    outputs = model.generate(inputs['input_ids'], max_length=150, num_return_sequences=1, no_repeat_ngram_size=2, top_p=0.95, top_k=60)

    # Decode the generated response
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Return the answer (after removing the question part)
    return answer.replace(question, "").strip()

# Test the model with a question
question = "What is your name?"
answer = ask_ai(question)
print(f"Answer: {answer}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Answer: I'm a guy who's been doing this for a long time. I'm not a big fan of the word "dude." I think it's a bad word.
.
